In [2]:
import pandas as pd
import random

# === [前提] 既存のP1〜P5データを読み込む（CSVファイルなどから）
# ※ 例：df_existing = pd.read_csv("existing_conditions.csv")
# 今はサンプルとして DataFrame に手動で入れる

data_existing = [
    ["P1", "空間参照だけ", "A", "ポインティングだけ", "B", "Pointing+空間参照", "C", "ラベル", "D", 1, 3, 2, 4],
    ["P2", "Pointing+空間参照", "B", "空間参照だけ", "C", "ラベル", "D", "ポインティングだけ", "A", 2, 1, 4, 3],
    ["P3", "ラベル", "C", "Pointing+空間参照", "D", "ポインティングだけ", "A", "空間参照だけ", "B", 4, 2, 3, 1],
    ["P4", "ポインティングだけ", "D", "ラベル", "A", "空間参照だけ", "B", "Pointing+空間参照", "C", 3, 4, 1, 2],
    ["P5", "空間参照だけ", "A", "ポインティングだけ", "B", "Pointing+空間参照", "C", "ラベル", "D", 1, 3, 2, 4],
]

columns = [
    "Participant",
    "Condition 1", "Task Set 1",
    "Condition 2", "Task Set 2",
    "Condition 3", "Task Set 3",
    "Condition 4", "Task Set 4",
    "Condition 1_ConditionNum",
    "Condition 2_ConditionNum",
    "Condition 3_ConditionNum",
    "Condition 4_ConditionNum"
]

df_existing = pd.DataFrame(data_existing, columns=columns)

# === [新規追加] P6〜P24 をラテン方格で生成 ===

conditions = ["空間参照だけ", "ポインティングだけ", "Pointing+空間参照", "ラベル"]
condition_to_num = {c: i+1 for i, c in enumerate(conditions)}
task_sets = ["A", "B", "C", "D"]
num_conditions = len(conditions)

def generate_balanced_latin_square(n):
    square = []
    for i in range(n):
        row = []
        for j in range(n):
            if j % 2 == 0:
                row.append((i + j//2) % n)
            else:
                row.append((n + i - (j+1)//2) % n)
        square.append(row)
    return square

latin_square = generate_balanced_latin_square(num_conditions)
task_set_orders = [task_sets[i:] + task_sets[:i] for i in range(num_conditions)]

new_rows = []
num_new = 24 - len(df_existing)
num_blocks = num_new // num_conditions

for block in range(num_blocks):
    shuffled_conditions = random.sample(latin_square, num_conditions)
    shuffled_tasks = random.sample(task_set_orders, num_conditions)
    for i in range(num_conditions):
        row = {
            "Participant": f"P{len(df_existing) + len(new_rows) + 1}"
        }
        for j in range(num_conditions):
            c_label = conditions[shuffled_conditions[i][j]]
            t_label = shuffled_tasks[i][j]
            row[f"Condition {j+1}"] = c_label
            row[f"Task Set {j+1}"] = t_label
            row[f"Condition {j+1}_ConditionNum"] = condition_to_num[c_label]
        new_rows.append(row)

df_new = pd.DataFrame(new_rows)

# === 結合して最終結果にする ===
df_all = pd.concat([df_existing, df_new], ignore_index=True)

# === 保存（任意） ===
df_all.to_csv("latin_square_extended.csv", index=False)
print(df_all.to_string(index=False))


Participant   Condition 1 Task Set 1   Condition 2 Task Set 2   Condition 3 Task Set 3   Condition 4 Task Set 4  Condition 1_ConditionNum  Condition 2_ConditionNum  Condition 3_ConditionNum  Condition 4_ConditionNum
         P1        空間参照だけ          A     ポインティングだけ          B Pointing+空間参照          C           ラベル          D                         1                         3                         2                         4
         P2 Pointing+空間参照          B        空間参照だけ          C           ラベル          D     ポインティングだけ          A                         2                         1                         4                         3
         P3           ラベル          C Pointing+空間参照          D     ポインティングだけ          A        空間参照だけ          B                         4                         2                         3                         1
         P4     ポインティングだけ          D           ラベル          A        空間参照だけ          B Pointing+空間参照          C                         

In [2]:
!pip install ace_tools


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import json
import os

# 対象フォルダとファイル接頭文字
folder_path = "../InteractiveSmartHome/Assets/EXPERIMENT/ArrangeData"
letters = ['A', 'B', 'C', 'D']

for letter in letters:
    file_name = f"PreTaskArrangement{letter}.json"
    file_path = os.path.join(folder_path, file_name)

    # ファイル読み込み
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # task_id の追加
    for idx, item in enumerate(data):
        item["task_id"] = f"T{letter}{idx}"

    # 保存先ファイル名
    output_file = f"PreTaskArrangement{letter}.json"
    output_path = os.path.join(folder_path, output_file)

    # 保存
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"✅ {output_file} を保存しました")


✅ PreTaskArrangementA.json を保存しました
✅ PreTaskArrangementB.json を保存しました
✅ PreTaskArrangementC.json を保存しました
✅ PreTaskArrangementD.json を保存しました


In [9]:
import csv

# CSVに書き込むデータ（1行目はヘッダー）
header = ['ConditionID', 'ConditionName']
rows = [
    [1, 'Spatial Reference (Wizard of Oz)'],
    [2, 'Label (Wizard of Oz)'],
    [3, 'Pointing (Wizard of Oz)'],
    [4, 'Spatial Reference (System)']
]

# ファイル出力
with open('conditions.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(header)
    writer.writerows(rows)

print("✅ conditions.csv が作成されました。")


✅ conditions.csv が作成されました。
